# Event Study — Geopolitical Shocks on EUR Yields

**Methodology:** Constant-Mean Model

**Events:**
- 🇺🇦 Ukraine invasion: **2022-02-23**
- 🇮🇷 Iran escalation (2026): **2026-02-27**

**Assets:** Bund yields (2Y/5Y/10Y), AAA Euro Area yields (2Y/5Y/10Y), 1Y1Y & 5Y5Y Forward (Computed), Slope 10Y-2Y, Brent Oil

---
> **Notebook structure:**
> 1. Setup & Data Ingestion
> 2. Exploratory plots (since Jan 2026)
> 3. Price plots around event dates
> 4. Event study (CAR)
> 5. Statistical inference




---
## 0. Install & Import

In [38]:
# ── Installs (run once) ──────────────────────────────────────────────────────
# !pip install pandas numpy scipy plotly requests --quiet

In [39]:
import os
import warnings
import requests
from io import StringIO

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import get_window
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings('ignore')
pio.renderers.default = 'notebook'   # change to 'browser' if not in JupyterLab

print('✓ Imports OK')

✓ Imports OK


---
## 1. Global Parameters

In [ ]:
# ── Time range ───────────────────────────────────────────────────────────────
START  = '2019-01-01'
END    = pd.Timestamp('today').strftime('%Y-%m-%d')
BDAYS  = pd.bdate_range(start=START, end=END)

# ── FRED API key  (set as env var: export FRED_API_KEY=xxx) ──────────────────
# NEVER hard-code credentials in notebooks that may be shared.
FRED_API_KEY = ""

# ── Local Bund CSV files  (ECB published data) ───────────────────────────────

BUND_DIR   = '/Users/travail/Documents/Adhoc Analysis/event study iran vs ukraine/Core/bund data'
BUND_FILES = {
    'bund2y':  os.path.join(BUND_DIR, 'BBSSY.D.REN.EUR.A610.000000WT0202.A.csv'),
    'bund5y':  os.path.join(BUND_DIR, 'BBSSY.D.REN.EUR.A620.000000WT0505.A.csv'),
    'bund10y': os.path.join(BUND_DIR, 'BBSSY.D.REN.EUR.A630.000000WT1010.A.csv'),
}

# ── Event study windows (business days relative to t0) ───────────────────────
W_EST = (-250, -11)   # estimation window
W_EVT = (-10,  +20)   # event window

# ── Events ───────────────────────────────────────────────────────────────────
EVENTS = {
    'Ukraine (23/02/2022)':        pd.Timestamp('2022-02-23'),
    'Iran Escalation (27/02/2026)': pd.Timestamp('2026-02-27'),
}

# ── Display metadata ─────────────────────────────────────────────────────────
LABELS = {
    'bund2y':           'Bund 2Y',
    'bund5y':           'Bund 5Y',
    'bund10y':          'Bund 10Y',
    'ea2Y':             'AAA EA 2Y',
    'ea5Y':             'AAA EA 5Y',
    'ea10Y':            'AAA EA 10Y',
    'fwd5y5y_bund':     '5Y5Y Fwd (Bund)',
    'fwd5y5y_ea':       '5Y5Y Fwd (AAA EA)',
    'slope_10y2y':      'Slope 10Y-2Y',
    'oil':              'Brent Oil ($/bbl)',
}

COLORS_EVENTS = {
    'Ukraine (23/02/2022)':        '#E63946',
    'Iran Escalation (27/02/2026)': '#457B9D',
}

# Palette for assets
COLORS_ASSETS = {
    'bund2y':       '#1d3557',
    'bund5y':       '#457b9d',
    'bund10y':      '#a8dadc',
    'ea2Y':         '#e63946',
    'ea5Y':         '#f4a261',
    'ea10Y':        '#e76f51',
    'fwd5y5y_bund': '#2a9d8f',
    'fwd5y5y_ea':   '#264653',
    'slope_10y2y':  '#8338ec',
    'oil':          '#fb8500',
}

ASSETS_YIELD = ['bund2y','bund5y','bund10y','ea2Y','ea5Y','ea10Y',
                'fwd5y5y_bund','fwd5y5y_ea','slope_10y2y']
ASSETS_ALL   = ASSETS_YIELD + ['oil']

print(f'✓ Parameters set  |  Study: {START} → {END}')

✓ Parameters set  |  Study: 2019-01-01 → 2026-03-19


---
## 2. Data Ingestion

In [ ]:
#  2-A.  ECB Data Portal REST API  (AAA Euro Area yield curve)
# ────────────────────────────────────────────────────────────────────────────

ECB_YC_SERIES = {
    'ea2Y':  'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y',
    'ea5Y':  'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y',
    'ea10Y': 'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y',
}

def fetch_ecb_yc(series_key: str, start=START, end=END):
    """Fetch a single ECB yield-curve series and return a business-day aligned Series."""
    flow, key = series_key.split('/', 1)
    url = f'https://data-api.ecb.europa.eu/service/data/{flow}/{key}'
    params = {'startPeriod': start, 'endPeriod': end,
              'format': 'csvdata', 'detail': 'dataonly'}
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text))

        # Normalise column names to find time and value columns
        col_t = next((c for c in df.columns if 'TIME' in c.upper() or 'DATE' in c.upper()), None)
        col_v = next((c for c in df.columns if 'VALUE' in c.upper() or 'OBS' in c.upper()), None)
        if not (col_t and col_v):
            print(f'  ✗ ECB {key}: unrecognised columns {df.columns.tolist()}')
            return None

        s = (df.set_index(pd.to_datetime(df[col_t]))[col_v]
               .astype(float).sort_index())
        s = s.loc[start:end]

        # Align to business days; forward-fill; drop leading-NaN rows
        s = s.reindex(BDAYS.union(s.index)).ffill().reindex(BDAYS)
        print(f'  ✓ ECB {key[-30:]:<30} | '
              f'{s.dropna().index[0].date()} → {s.dropna().index[-1].date()} '
              f'({s.dropna().shape[0]} obs)')
        return s
    except requests.HTTPError as e:
        print(f'  ✗ ECB {key}: HTTP {e.response.status_code}')
    except Exception as e:
        print(f'  ✗ ECB {key}: {type(e).__name__}: {e}')
    return None


print('Fetching ECB AAA Euro Area yield curve...')
ea2Y  = fetch_ecb_yc(ECB_YC_SERIES['ea2Y'])
ea5Y  = fetch_ecb_yc(ECB_YC_SERIES['ea5Y'])
ea10Y = fetch_ecb_yc(ECB_YC_SERIES['ea10Y'])

Fetching ECB AAA Euro Area yield curve...
  ✓ ECB .U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y | 2019-01-02 → 2026-03-19 (1882 obs)
  ✓ ECB .U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y | 2019-01-02 → 2026-03-19 (1882 obs)
  ✓ ECB U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y | 2019-01-02 → 2026-03-19 (1882 obs)


In [ ]:
#  2-B.  Bundesbank Bund CSV data
# ────────────────────────────────────────────────────────────────────────────

def clean_bund_csv(path: str) -> pd.Series:
    """
    Read an Bundesbank published Bund CSV (first 4 rows are metadata).
    Returns a business-day aligned Series of float yields.
    """
    raw = pd.read_csv(path, skiprows=4, header=0)
    raw.columns = ['Date', 'Value'] + list(raw.columns[2:])
    raw = raw[['Date', 'Value']].copy()
    raw['Date']  = pd.to_datetime(raw['Date'], errors='coerce')
    raw['Value'] = pd.to_numeric(raw['Value'], errors='coerce')
    raw = (raw.dropna(subset=['Date', 'Value'])
              .query('@START <= Date <= @END')
              .sort_values('Date')
              .drop_duplicates('Date'))
    s = pd.Series(raw['Value'].values, index=raw['Date'])
    s = s.reindex(BDAYS.union(s.index)).ffill().reindex(BDAYS)
    print(f'  ✓ {os.path.basename(path):<50} | '
          f'{s.dropna().index[0].date()} → {s.dropna().index[-1].date()} '
          f'({s.dropna().shape[0]} obs)')
    return s


print('Reading Bund CSV files...')
bund2y  = clean_bund_csv(BUND_FILES['bund2y'])
bund5y  = clean_bund_csv(BUND_FILES['bund5y'])
bund10y = clean_bund_csv(BUND_FILES['bund10y'])

Reading Bund CSV files...
  ✓ BBSSY.D.REN.EUR.A610.000000WT0202.A.csv            | 2019-01-02 → 2026-03-19 (1882 obs)
  ✓ BBSSY.D.REN.EUR.A620.000000WT0505.A.csv            | 2019-01-02 → 2026-03-19 (1882 obs)
  ✓ BBSSY.D.REN.EUR.A630.000000WT1010.A.csv            | 2019-01-02 → 2026-03-19 (1882 obs)


In [ ]:
#  2-C.  FRED — Brent Oil
# ────────────────────────────────────────────────────────────────────────────

def fetch_fred(series_id: str, start=START, end=END):
    """Fetch a FRED series and return a business-day aligned Series."""
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {'series_id': series_id, 'api_key': FRED_API_KEY,
              'file_type': 'json',
              'observation_start': start, 'observation_end': end}
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        obs = [(pd.Timestamp(o['date']), float(o['value']))
               for o in data.get('observations', [])
               if o['value'] != '.']
        if not obs:
            print(f'  ✗ FRED {series_id}: empty')
            return None
        s = pd.Series([v for _, v in obs], index=[d for d, _ in obs])
        s = s[~s.index.duplicated()].sort_index()
        s = s.reindex(BDAYS.union(s.index)).ffill().reindex(BDAYS)
        print(f'  ✓ FRED {series_id} | '
              f'{s.dropna().index[0].date()} → {s.dropna().index[-1].date()} '
              f'({s.dropna().shape[0]} obs)')
        return s
    except Exception as e:
        print(f'  ✗ FRED {series_id}: {type(e).__name__}: {e}')
    return None


print('Fetching Brent Oil from FRED...')
oil = fetch_fred('DCOILBRENTEU')

Fetching Brent Oil from FRED...
  ✓ FRED DCOILBRENTEU | 2019-01-02 → 2026-03-19 (1882 obs)


In [ ]:
import yfinance as yf

# ────────────────────────────────────────────────────────────────────────────
# Fetch Brent crude price from Yahoo Finance (ticker: BZ=F) and align to business days
# ────────────────────────────────────────────────────────────────────────────

ticker = 'BZ=F'   # Brent Crude Futures continuous contract
yahoo_df = yf.download(ticker, start=START, end=END, progress=False)

# Use adjusted close (or close if adj close missing), align to BDAYS and forward-fill
oil_yahoo = yahoo_df['Close'].fillna(yahoo_df['Close']).copy()
oil_yahoo.index = pd.to_datetime(oil_yahoo.index)
oil_yahoo = oil_yahoo.reindex(BDAYS.union(oil_yahoo.index)).ffill().reindex(BDAYS)
oil_yahoo.name = 'oil_yahoo'

print(f'✓ Yahoo Brent series: {oil_yahoo.dropna().index[0].date()} → {oil_yahoo.dropna().index[-1].date()} '
      f'({oil_yahoo.dropna().shape[0]} obs)')

# Ensure `oil_yahoo` is a 1‑D Series aligned to business days
if isinstance(oil_yahoo, pd.DataFrame):
    oil_yahoo = oil_yahoo.squeeze() # If DataFrame with single column, convert to Series
    oil_yahoo.name = 'oil_yahoo'

oil_yahoo = oil_yahoo.reindex(BDAYS).ffill()

oil_yahoo.tail(10)

✓ Yahoo Brent series: 2019-01-02 → 2026-03-19 (1882 obs)


2026-03-06     92.690002
2026-03-09     98.959999
2026-03-10     87.800003
2026-03-11     91.980003
2026-03-12    100.459999
2026-03-13    103.139999
2026-03-16    100.209999
2026-03-17    103.419998
2026-03-18    107.379997
2026-03-19    107.379997
Freq: B, Name: oil_yahoo, dtype: float64

In [45]:
oil = oil_yahoo

In [46]:
# ────────────────────────────────────────────────────────────────────────────
#  2-D.  Derived series
# ────────────────────────────────────────────────────────────────────────────

def fwd_5y5y(r5: pd.Series, r10: pd.Series) -> pd.Series:
    """
    Nominal 5Y5Y forward rate via discrete compounding:
        f = [ (1 + r10)^10 / (1 + r5)^5 ]^(1/5) - 1
    Inputs in % (e.g. 3.5 for 3.5%). Returns in %.
    """
    r5_  = r5.astype(float)  / 100
    r10_ = r10.astype(float) / 100
    f    = (((1 + r10_) ** 10 / (1 + r5_) ** 5) ** 0.2 - 1) * 100
    return f


fwd5y5y_bund = fwd_5y5y(bund5y, bund10y).rename('fwd5y5y_bund')
fwd5y5y_ea   = fwd_5y5y(ea5Y,   ea10Y  ).rename('fwd5y5y_ea')
slope_10y2y  = (ea10Y - ea2Y).rename('slope_10y2y')

print('✓ Derived series computed')

✓ Derived series computed


In [ ]:
#  2-E.  Master DataFrame
# ────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame({
    'bund2y':       bund2y,
    'bund5y':       bund5y,
    'bund10y':      bund10y,
    'ea2Y':         ea2Y,
    'ea5Y':         ea5Y,
    'ea10Y':        ea10Y,
    'fwd5y5y_bund': fwd5y5y_bund,
    'fwd5y5y_ea':   fwd5y5y_ea,
    'slope_10y2y':  slope_10y2y,
    'oil':          oil,
}, index=BDAYS)

# Forward-fill residual gaps (holidays, etc.) then trim to 2020
df = df.ffill()
df_full = df.copy()          # keep full history for event study
df_2020 = df.loc['2020':].copy()

print(f'✓ Master DataFrame  |  Shape: {df.shape}')
print(f'  Date range : {df.index[0].date()} → {df.index[-1].date()}')
print(f'  Columns    : {df.columns.tolist()}')
print()
print(df[['bund2y','bund5y','bund10y','fwd5y5y_ea','slope_10y2y','oil']]
      .describe().round(3))

✓ Master DataFrame  |  Shape: (1883, 10)
  Date range : 2019-01-01 → 2026-03-19
  Columns    : ['bund2y', 'bund5y', 'bund10y', 'ea2Y', 'ea5Y', 'ea10Y', 'fwd5y5y_bund', 'fwd5y5y_ea', 'slope_10y2y', 'oil']

         bund2y    bund5y   bund10y  fwd5y5y_ea  slope_10y2y       oil
count  1882.000  1882.000  1882.000    1882.000     1882.000  1882.000
mean      0.906     0.920     1.140       1.476        0.318    72.515
std       1.536     1.416     1.361       1.371        0.406    17.636
min      -1.020    -0.960    -0.830      -0.634       -0.644    19.330
25%      -0.680    -0.600    -0.310       0.048        0.062    63.355
50%       0.870     1.180     1.440       1.834        0.373    72.695
75%       2.260     2.290     2.470       2.675        0.644    82.960
max       3.340     2.860     2.970       3.513        0.951   127.980


---
## LAST MINUTE ADDITION

In [ ]:
#  Add ECB ESTR (Euro Short-Term Rate) in asset series (SDW series key)
# ────────────────────────────────────────────────────────────────────────────

def fetch_ecb_sdw(series_key: str, start=START, end=END):
    """
    Fetch an ECB SDW series (by full series key) and return a business-day aligned Series.
    Example key: 'EST.B.EU000A2X2A25.WT'
    """
    flow, key = series_key.split('.', 1)
    url = f'https://data-api.ecb.europa.eu/service/data/{flow}/{key}'
    params = {'startPeriod': start, 'endPeriod': end,
              'format': 'csvdata', 'detail': 'dataonly'}
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text))

        col_t = next((c for c in df.columns if 'TIME' in c.upper() or 'DATE' in c.upper()), None)
        col_v = next((c for c in df.columns if 'VALUE' in c.upper() or 'OBS' in c.upper()), None)
        if not (col_t and col_v):
            print(f'  ✗ ECB {series_key}: unrecognised columns {df.columns.tolist()}')
            return None

        s = (df.set_index(pd.to_datetime(df[col_t]))[col_v]
               .astype(float).sort_index())
        s = s.loc[start:end]
        s = s.reindex(BDAYS.union(s.index)).ffill().reindex(BDAYS)

        print(f'  ✓ ECB {series_key} | {s.dropna().index[0].date()} → {s.dropna().index[-1].date()} '
              f'({s.dropna().shape[0]} obs)')
        return s
    except Exception as e:
        print(f'  ✗ ECB {series_key}: {type(e).__name__}: {e}')
        return None


# Use the requested series key
ESTR_SERIES_KEY = 'EST.B.EU000A2X2A25.WT'
estr = fetch_ecb_sdw(ESTR_SERIES_KEY)

# Add to asset lists / labels / colors
LABELS['estr'] = 'ECB ESTR (%)'
COLORS_ASSETS['estr'] = '#2c7fb8'

# if 'estr' not in ASSETS_YIELD:
#     ASSETS_YIELD.append('estr')
# if 'estr' not in ASSETS_ALL:
#     ASSETS_ALL.append('estr')

# If master df already exists, append ESTR into it (and refresh derived subsets)
if 'df' in globals():
    df = df.copy()
    df['estr'] = estr
    df = df.ffill()
    df_full = df.copy()
    df_2020 = df.loc['2020':].copy()
    print('✓ ESTR added to master DataFrame')

  ✓ ECB EST.B.EU000A2X2A25.WT | 2019-10-01 → 2026-03-19 (1688 obs)
✓ ESTR added to master DataFrame


In [49]:
# ────────────────────────────────────────────────────────────────────────────
#  Scrape EURIBOR (all tenors) from ECB SDW
# ────────────────────────────────────────────────────────────────────────────

#EURIBOR_SERIES = {
#    'Euribor 1W':  'IR/EURIBOR1WD',
#    'Euribor 1M':  'IR/EURIBOR1MD',
#    'Euribor 3M':  'IR/EURIBOR3MD',
#    'Euribor 6M':  'IR/EURIBOR6MD',
#    'Euribor 12M': 'IR/EURIBOR12MD',
#}#

#euribor_data = {}
#for name, series_key in EURIBOR_SERIES.items():
#    s = fetch_ecb_sdw(series_key, start=START, end=END)
#    if s is not None:
#        euribor_data[name] = s
#
#euribor_df = pd.DataFrame(euribor_data).reindex(BDAYS).ffill()
#
#print('✓ EURIBOR series fetched:')
#print(euribor_df.iloc[:5].to_string())

In [ ]:
# Add ECB 1Y yield, then compute 1Y1Y forward rates
# ────────────────────────────────────────────────────────────────────────────

# Add to ECB series
ECB_YC_SERIES['ea1Y'] = 'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y'
ea1Y = fetch_ecb_yc(ECB_YC_SERIES['ea1Y'])

# Define 1Y1Y forward rate function
def fwd_1y1y(r1: pd.Series, r2: pd.Series) -> pd.Series:
    """
    1Y1Y forward rate via discrete compounding:
        f = [ (1 + r2)^2 / (1 + r1) ] - 1
    Inputs in % (e.g. 3.5 for 3.5%). Returns in %.
    """
    r1_ = r1.astype(float) / 100
    r2_ = r2.astype(float) / 100
    f = (((1 + r2_) ** 2 / (1 + r1_) - 1) * 100)
    return f

fwd1y1y_ea = fwd_1y1y(ea1Y, ea2Y)
fwd1y1y_ea = fwd1y1y_ea.rename('fwd1y1y_ea')

# Add to asset lists / labels / colors
LABELS['ea1Y'] = 'AAA EA 1Y'
LABELS['fwd1y1y_ea'] = '1Y1Y Fwd (AAA EA)'

COLORS_ASSETS['ea1Y'] = '#e63946'
COLORS_ASSETS['fwd1y1y_ea'] = '#264653'

if 'ea1Y' not in ASSETS_YIELD:
    ASSETS_YIELD.append('ea1Y')
    ASSETS_YIELD.append('bund1y')
    ASSETS_YIELD.append('fwd1y1y_bund')
if 'fwd1y1y_ea' not in ASSETS_YIELD:
    ASSETS_YIELD.append('fwd1y1y_ea')

if 'ea1Y' not in ASSETS_ALL:
    ASSETS_ALL.append('ea1Y')

if 'fwd1y1y_ea' not in ASSETS_ALL:
    ASSETS_ALL.append('fwd1y1y_ea')

# If master df already exists, append into it (and refresh derived subsets)
if 'df' in globals():
    df = df.copy()
    df['ea1Y'] = ea1Y
    df['fwd1y1y_ea'] = fwd1y1y_ea
    df = df.ffill()
    df_full = df.copy()
    df_2020 = df.loc['2020':].copy()
    print('✓ 1Y1Y forward series added to master DataFrame')

  ✓ ECB .U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y | 2019-01-02 → 2026-03-19 (1882 obs)
✓ 1Y1Y forward series added to master DataFrame


---
## 3. Exploratory Plots — Asset Levels Since January 2026

In [52]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
    x=oil.loc['2021-10-01':].index,
    y=oil.loc['2021-10-01':].values,
    mode='lines',
    name=LABELS['oil'],
    line=dict(color=COLORS_ASSETS['oil'], width=2),
))

for evt_label, t0 in EVENTS.items():
    if oil.loc['2021-10-01':].index.min() <= t0 <= oil.loc['2026-01-01':].index.max():
        fig.add_vline(
            x=t0.timestamp() * 1000,
            line=dict(color=COLORS_EVENTS[evt_label], width=2, dash='dash'),
            annotation_text=evt_label,
            annotation_position='top right' if 'Ukraine' in evt_label else 'top left',
            annotation_font=dict(size=11, color=COLORS_EVENTS[evt_label]),
        )

fig.update_layout(
    title=dict(text='Brent Oil Price — with Ukraine & Iran Shock Dates', font=dict(size=16)),
    xaxis_title='Date',
    yaxis_title='Brent Oil ($/bbl)',
    template='plotly_white',
    hovermode='x unified',
)

#fig.write_html('plot_oil_with_war_markers.html')
fig.show()
print('✓ Plot saved → plot_oil_with_war_markers.html')

✓ Plot saved → plot_oil_with_war_markers.html


In [53]:
from plotly.subplots import make_subplots

# ────────────────────────────────────────────────────────────────────────────
#  Plot: Oil Price — Left: Around Iran Escalation (±40 days), Right: Full History since Oct 2021
# ────────────────────────────────────────────────────────────────────────────

import plotly.graph_objects as go

def get_window(df: pd.DataFrame, t0: pd.Timestamp,
                     lo: int, hi: int) -> pd.DataFrame:
    """
    Return rows of df in [t0 + lo, ..., t0 + hi] business days.
    Also adds a 'rel_day' column (integer offset from t0).
    """
    idx = df.index
    pos = idx.searchsorted(t0)
    i_lo = max(0, pos + lo)
    i_hi = min(len(idx) - 1, pos + hi)
    win  = df.iloc[i_lo : i_hi + 1].copy()
    win['rel_day'] = np.arange(i_lo - pos, i_hi - pos + 1)
    return win

# Define the event for Iran escalation
evt_label = 'Iran Escalation (27/02/2026)'
t0 = EVENTS[evt_label]
WINDOW_PLOT = (-25, 40)

# Get the window around the event
win = get_window(df_full, t0, *WINDOW_PLOT)

# Create subplots: left for event window, right for full history
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.80, 0.20],
    subplot_titles=['Oil Price since Oct 2021', 'Oil Price around Iran Escalation'],
    horizontal_spacing=0.1,
)

# Right panel: Oil around Iran event
fig.add_trace(
    go.Scatter(
        x=win['rel_day'],
        y=win['oil'],
        mode='lines',
        name='Oil around Iran',
        line=dict(color=COLORS_ASSETS['oil'], width=2),
    ),
    row=1, col=2
)
# Add vertical line at event day (rel_day=0)
fig.add_vline(x=0, line=dict(color=COLORS_EVENTS[evt_label], width=2, dash='dash'), row=1, col=1)

# Left panel: Full oil history since 2021-10-01
fig.add_trace(
    go.Scatter(
        x=oil.loc['2021-10-01':].index,
        y=oil.loc['2021-10-01':].values,
        mode='lines',
        name='Oil full history',
        line=dict(color=COLORS_ASSETS['oil'], width=2),
    ),
    row=1, col=1
)
# Add vertical lines for events
for evt_label_loop, t0_loop in EVENTS.items():
    if oil.loc['2021-10-01':].index.min() <= t0_loop <= oil.loc['2026-01-01':].index.max():
        fig.add_vline(
            x=t0_loop.timestamp() * 1000,
            line=dict(color=COLORS_EVENTS[evt_label_loop], width=2, dash='dash'),
            annotation_text=evt_label_loop,
            annotation_position='top right' if 'Ukraine' in evt_label_loop else 'top left',
            annotation_font=dict(size=11, color=COLORS_EVENTS[evt_label_loop]),
            row=1, col=1
        )

# Update layout
fig.update_layout(
    title=dict(text='Brent Oil Price — Full History vs Focus Window', font=dict(size=16)),
    xaxis_title='Date',
    xaxis2_title='Business days relative to event',
    yaxis_title='Brent Oil ($/bbl)',
    yaxis2_title='Brent Oil ($/bbl)',
    template='plotly_white',
    hovermode='x unified',
    showlegend=False
)

# Save and show
fname = 'plot_oil_event_vs_full.html'
#fig.write_html(fname)
fig.show()
print(f'✓ Plot saved → {fname}')

✓ Plot saved → plot_oil_event_vs_full.html


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  Modified Plot 2 for Oil — with inset focus on post-event price increase
# ────────────────────────────────────────────────────────────────────────────

from plotly.subplots import make_subplots
import plotly.graph_objects as go

asset = 'oil'
WINDOW_PLOT = (-20, 20)

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=['Full Window (20 days)', 'Focus: Post-Event Increase (0 to +20 days)'],
    horizontal_spacing=0.05,
)

# Main plot (left)
for evt_label, t0 in EVENTS.items():
    color = COLORS_EVENTS[evt_label]
    win = get_window(df_full, t0, *WINDOW_PLOT)
    if asset not in win.columns:
        continue
    s = win[asset].copy()
    level_at_t0 = win.loc[win['rel_day'] == 0, asset]
    if level_at_t0.empty:
        closest = win.iloc[(win['rel_day'].abs()).argsort()[:1]]
        level_at_t0 = closest[asset].values[0]
    else:
        level_at_t0 = level_at_t0.iloc[0]
    y_norm = s.values - level_at_t0
    hover_dates = [f'{d.strftime("%d %b %Y")}' for d in win.index]
    fig.add_trace(go.Scatter(
        x=win['rel_day'],
        y=y_norm,
        mode='lines+markers',
        name=evt_label,
        line=dict(color=color, width=2.5),
        marker=dict(size=3.5),
        customdata=np.stack([hover_dates, s.values], axis=-1),
        hovertemplate=(
            '<b>' + evt_label + '</b><br>'
            'Day %{x}  (%{customdata[0]})<br>'
            'Level: %{customdata[1]:.3f}<br>'
            'Δ from t₀: %{y:.2f}'
            '<extra></extra>'
        ),
    ), row=1, col=1)

# Reference lines for main plot
fig.add_hline(y=0, line=dict(color='black', width=1, dash='dash'), row=1, col=1)
fig.add_vline(x=0, line=dict(color='gray', width=1.5, dash='dash'), row=1, col=1)

# Inset focus (right) — only for Iran escalation, post-event
evt_label = 'Iran Escalation (27/02/2026)'
t0 = EVENTS[evt_label]
win_post = get_window(df_full, t0, 0, 20)  # 0 to +20 days
if asset in win_post.columns:
    s_post = win_post[asset].copy()
    level_at_t0 = win_post.loc[win_post['rel_day'] == 0, asset].iloc[0]
    y_norm_post = s_post.values - level_at_t0
    hover_dates_post = [f'{d.strftime("%d %b %Y")}' for d in win_post.index]
    fig.add_trace(go.Scatter(
        x=win_post['rel_day'],
        y=y_norm_post,
        mode='lines+markers',
        name=f'{evt_label} (Focus)',
        line=dict(color=COLORS_EVENTS[evt_label], width=3),
        marker=dict(size=5),
        customdata=np.stack([hover_dates_post, s_post.values], axis=-1),
        hovertemplate=(
            '<b>' + evt_label + ' (Focus)</b><br>'
            'Day %{x}  (%{customdata[0]})<br>'
            'Level: %{customdata[1]:.3f}<br>'
            'Δ from t₀: %{y:.2f}'
            '<extra></extra>'
        ),
        showlegend=False
    ), row=1, col=2), 

# Reference lines for inset
fig.add_hline(y=0, line=dict(color='black', width=1, dash='dash'), row=1, col=2)
fig.add_vline(x=0, line=dict(color='gray', width=1.5, dash='dash'), row=1, col=2)

unit_label = 'Change from event-day level (log-ret ×100)'
fig.update_layout(
    title=dict(
        text=f'<b>{LABELS.get(asset, asset)}</b> — Ukraine Invasion vs Iran Escalation<br>'
             f'<sup>Left: Both series rebased to 0 at event date (±40 days). Right: Iran post-event focus (0 to +20 days)</sup>',
        font=dict(size=15),
    ),
    xaxis=dict(title='Business days relative to event (0 = event date)'),
    xaxis2=dict(title='Days post-event'),
    yaxis=dict(title=unit_label),
    yaxis2=dict(title=unit_label),
    height=560,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=1.02, x=0),
)

slug = asset.replace('/', '_')
fname = f'plot2_war_comparison_{slug}_with_inset.html'
fig.write_html(fname)
fig.show()
print(f'✓ Modified plot saved → {fname}')

✓ Modified plot saved → plot2_war_comparison_oil_with_inset.html


In [55]:
# ────────────────────────────────────────────────────────────────────────────
#  Plot 1 — EUR Yields since Jan 2026  (2x3 subplot grid)
# ────────────────────────────────────────────────────────────────────────────

df_2026 = df.loc['2026-01-01':].copy()

yield_groups = {
    'Bund Yields (%)':        ['bund2y',  'bund5y',  'bund10y'],
    'AAA Euro Area Yields (%)':['ea2Y',    'ea5Y',    'ea10Y'],
    'Slope 10Y-2Y (bp)':       ['slope_10y2y'],
    'Brent Oil ($/bbl)':   ['oil'],
    '5Y5Y Forward Rate (%)':   ['fwd5y5y_bund', 'fwd5y5y_ea'],
    #'ECB ESTR (%)':   ['estr'],
    '1Y1Y Forward Rate (%)': ['fwd1y1y_ea'],
}

n_rows = 2
fig = make_subplots(
    rows=n_rows, cols=3,
    subplot_titles=list(yield_groups.keys()),
    vertical_spacing=0.10,
    horizontal_spacing=0.07,
)

positions = [(1,1),(1,2),(1,3),(2,1),(2,2), (2,3)]

for pos, (panel_title, asset_list) in zip(positions, yield_groups.items()):
    row, col = pos
    for asset in asset_list:
        if asset not in df_2026.columns:
            continue
        s = df_2026[asset].dropna()
        fig.add_trace(
            go.Scatter(
                x=s.index, y=s.values,
                mode='lines',
                name=LABELS[asset],
                line=dict(color=COLORS_ASSETS[asset], width=2),
                showlegend=True,
                legendgroup=panel_title,
            ),
            row=row, col=col
        )
    # Mark Iran escalation event (27-Feb-2026)
    for t0_date in EVENTS.values():
        if df_2026.index.min() <= t0_date <= df_2026.index.max():
            fig.add_vline(
                x=t0_date.timestamp() * 1000,
                line=dict(color='#E63946', width=1.2, dash='dot'),
                row=row, col=col
            )

fig.update_layout(
    title=dict(
        text='EUR Rates & Brent Oil — January 2026 to Present',
        font=dict(size=18)
    ),
    height=950,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.07, x=0),
)

fig.write_html('plot1_asset_levels_2026.html')
fig.show()
print('✓ Plot 1 saved → plot1_asset_levels_2026.html')

✓ Plot 1 saved → plot1_asset_levels_2026.html


---
## 4. Price Plots Around Event Dates  *(±40 trading days)*

In [ ]:
#  Helper — extract window of ±N bdays around t0
# ────────────────────────────────────────────────────────────────────────────

def get_window(df: pd.DataFrame, t0: pd.Timestamp,
               lo: int, hi: int) -> pd.DataFrame:
    """
    Return rows of df in [t0 + lo, t0 + hi] business days.
    Also adds a 'rel_day' column (integer offset from t0).
    """
    idx = df.index
    pos = idx.searchsorted(t0)
    i_lo = max(0, pos + lo)
    i_hi = min(len(idx) - 1, pos + hi)
    win  = df.iloc[i_lo : i_hi + 1].copy()
    win['rel_day'] = np.arange(i_lo - pos, i_hi - pos + 1)
    return win

In [ ]:
#  Plot 2 — Raw yield levels around each event  (one figure per event)
# ────────────────────────────────────────────────────────────────────────────

WINDOW_PLOT = (-40, 40)    # ±40 bdays for the visual context window

for evt_label, t0 in EVENTS.items():
    win = get_window(df_full, t0, *WINDOW_PLOT)
    color = COLORS_EVENTS[evt_label]

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Bund Yields (%)',
            'AAA Euro Area Yields (%)',
            #'5Y5Y Forward Rate (%)',
            #'Slope 10Y-2Y (bp)  &  WTI Oil',
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
        specs=[[{},{}],[{},{'secondary_y': True}]]
    )

    # ── Panel 1: Bund ────────────────────────────────────────────────────────
    for asset in ['bund2y','bund5y','bund10y']:
        s = win[asset].dropna()
        fig.add_trace(go.Scatter(
            x=win['rel_day'], y=s.values,
            mode='lines', name=LABELS[asset],
            line=dict(color=COLORS_ASSETS[asset], width=2),
            legendgroup='bund'), row=1, col=1)

    # ── Panel 2: AAA EA ──────────────────────────────────────────────────────
    for asset in ['ea2Y','ea5Y','ea10Y']:
        s = win[asset].dropna()
        fig.add_trace(go.Scatter(
            x=win['rel_day'], y=s.values,
            mode='lines', name=LABELS[asset],
            line=dict(color=COLORS_ASSETS[asset], width=2),
            legendgroup='ea'), row=1, col=2)

    # ── Panel 3: 5Y5Y Forwards ───────────────────────────────────────────────
    for asset in ['fwd5y5y_bund','fwd5y5y_ea']:
        s = win[asset].dropna()
        fig.add_trace(go.Scatter(
            x=win['rel_day'], y=s.values,
            mode='lines', name=LABELS[asset],
            line=dict(color=COLORS_ASSETS[asset], width=2),
            legendgroup='fwd'), row=2, col=1)

    # ── Panel 4: Slope (left) + Oil (right) ──────────────────────────────────
    fig.add_trace(go.Scatter(
        x=win['rel_day'], y=win['slope_10y2y'].values,
        mode='lines', name=LABELS['slope_10y2y'],
        line=dict(color=COLORS_ASSETS['slope_10y2y'], width=2),
        legendgroup='slope'), row=2, col=2, secondary_y=False)

    if 'oil' in win.columns:
        fig.add_trace(go.Scatter(
            x=win['rel_day'], y=win['oil'].values,
            mode='lines', name=LABELS['oil'],
            line=dict(color=COLORS_ASSETS['oil'], width=2, dash='dot'),
            legendgroup='oil'), row=2, col=2, secondary_y=True)

    # ── Event line at rel_day = 0 ─────────────────────────────────────────────
    for r, c in [(1,1),(1,2),(2,1),(2,2)]:
        fig.add_vline(x=0, line=dict(color=color, width=2, dash='dash'),
                      row=r, col=c)

    fig.update_layout(
        title=dict(
            text=f'Yield Levels ±40 Days — <b>{evt_label}</b>  (day 0 = event)',
            font=dict(size=16)
        ),
        height=750,
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', y=-0.10, x=0),
    )
    fig.update_xaxes(title_text='Business days relative to event')

    slug = evt_label.replace(' ','_').replace('/','').replace('(','').replace(')','')[:30]
    fname = f'plot2_levels_{slug}.html'
    fig.write_html(fname)
    fig.show()
    print(f'✓ Plot 2 saved → {fname}')

✓ Plot 2 saved → plot2_levels_Ukraine_23022022.html


✓ Plot 2 saved → plot2_levels_Iran_Escalation_27022026.html


In [ ]:
#  Plot 2 — One figure per asset, both wars overlaid
#           x-axis = business days relative to event (day 0 = shock date)
#           y-axis = level (yield % or oil $/bbl), normalised to day 0 = 0
#                    so you compare *changes from shock* not absolute levels
# ────────────────────────────────────────────────────────────────────────────

WINDOW_PLOT = (-40, 40)   # ±40 business days around each event

for asset in ASSETS_ALL:
    if asset not in df_full.columns:
        continue

    fig = go.Figure()

    for evt_label, t0 in EVENTS.items():
        color = COLORS_EVENTS[evt_label]

        # Extract ±40 bday window around this event
        win = get_window(df_full, t0, *WINDOW_PLOT)

        if asset not in win.columns:
            continue

        s = win[asset].copy()

        # Normalise: subtract the level at day 0 so both series start at 0
        # This makes cross-event comparison of *changes* clean
        level_at_t0 = win.loc[win['rel_day'] == 0, asset]
        if level_at_t0.empty:
            # If exact day 0 missing (holiday), use closest available
            closest = win.iloc[(win['rel_day'].abs()).argsort()[:1]]
            level_at_t0 = closest[asset].values[0]
        else:
            level_at_t0 = level_at_t0.iloc[0]

        y_norm = s.values - level_at_t0

        # ── Actual dates as hover text ────────────────────────────────────────
        hover_dates = [
            f'{d.strftime("%d %b %Y")}' for d in win.index
        ]

        # ── Filled area (light) ───────────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=win['rel_day'].tolist() + win['rel_day'].tolist()[::-1],
            y=np.where(y_norm >= 0, y_norm, 0).tolist() +
              [0] * len(y_norm),
            fill='toself',
            #fillcolor=f'{color}',   # very transparent
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip',
        ))

        # ── Main line ─────────────────────────────────────────────────────────
        unit = 'log-ret×100' if asset == 'oil' else 'bp'
        fig.add_trace(go.Scatter(
            x=win['rel_day'],
            y=y_norm,
            mode='lines+markers',
            name=evt_label,
            line=dict(color=color, width=2.5),
            marker=dict(size=3.5),
            customdata=np.stack([hover_dates, s.values], axis=-1),
            hovertemplate=(
                '<b>' + evt_label + '</b><br>'
                'Day %{x}  (%{customdata[0]})<br>'
                f'Level: %{{customdata[1]:.3f}}<br>'
                f'Δ from t₀: %{{y:.2f}}'
                '<extra></extra>'
            ),
        ))

    # ── Reference lines ───────────────────────────────────────────────────────
    fig.add_hline(y=0, line=dict(color='black', width=1, dash='dash'))
    fig.add_vline(x=0, line=dict(color='gray',  width=1.5, dash='dash'),
                  annotation_text='Event day',
                  annotation_position='top right',
                  annotation_font=dict(size=11, color='gray'))

    # ── Annotations: pre / post zones ─────────────────────────────────────────
    fig.add_vrect(
        x0=WINDOW_PLOT[0], x1=0,
        fillcolor='rgba(200,200,200,0.06)', line_width=0,
        annotation_text='Pre-event', annotation_position='top left',
        annotation_font=dict(size=10, color='gray'),
    )
    fig.add_vrect(
        x0=0, x1=WINDOW_PLOT[1],
        fillcolor='rgba(255,220,180,0.06)', line_width=0,
        annotation_text='Post-event', annotation_position='top right',
        annotation_font=dict(size=10, color='gray'),
    )

    unit_label = 'Change from event-day level (log-ret ×100)' \
                 if asset == 'oil' \
                 else 'Change from event-day level (percentage points)'

    fig.update_layout(
        title=dict(
            text=(f'<b>{LABELS.get(asset, asset)}</b> — '
                  f'Ukraine Invasion vs Iran Escalation<br>'
                  f'<sup>Both series rebased to 0 at the event date — '
                  f'x-axis = business days relative to shock</sup>'),
            font=dict(size=15),
        ),
        xaxis=dict(
            title='Business days relative to event (0 = event date)',
            zeroline=False,
        ),
        yaxis=dict(title=unit_label),
        height=460,
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            orientation='h',
            yanchor='bottom', y=1.02,
            xanchor='right',  x=1,
        ),
    )

    slug  = asset.replace('/', '_')
    fname = f'plot2_war_comparison_{slug}.html'
    fig.write_html(fname)
    fig.show()
    print(f'✓  {LABELS.get(asset, asset):<30} → {fname}')

✓  Bund 2Y                        → plot2_war_comparison_bund2y.html


✓  Bund 5Y                        → plot2_war_comparison_bund5y.html


✓  Bund 10Y                       → plot2_war_comparison_bund10y.html


✓  AAA EA 2Y                      → plot2_war_comparison_ea2Y.html


✓  AAA EA 5Y                      → plot2_war_comparison_ea5Y.html


...

✓  5Y5Y Fwd (AAA EA)              → plot2_war_comparison_fwd5y5y_ea.html


✓  Slope 10Y-2Y                   → plot2_war_comparison_slope_10y2y.html


✓  Brent Oil ($/bbl)              → plot2_war_comparison_oil.html


✓  AAA EA 1Y                      → plot2_war_comparison_ea1Y.html


✓  1Y1Y Fwd (AAA EA)              → plot2_war_comparison_fwd1y1y_ea.html


---
## 5. Event Study — Constant Mean Model

In [ ]:
#  5-A.  Core functions
# ────────────────────────────────────────────────────────────────────────────

def get_biz_window_idx(t0: pd.Timestamp, lo: int, hi: int,
                       idx: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """Return the DatetimeIndex slice [t0+lo, ..., t0+hi] in business days."""
    pos  = idx.searchsorted(t0)
    i_lo = max(0, pos + lo)
    i_hi = min(len(idx) - 1, pos + hi)
    return idx[i_lo : i_hi + 1]


def run_event_study(df: pd.DataFrame, t0: pd.Timestamp,
                    assets: list[str],
                    w_est=W_EST, w_evt=W_EVT) -> dict:
    """
    Constant-mean event study for a single event.

    Model:  r_it = μ_i + ε_it
    AR_t  = r_it - μ_i          (abnormal return)
    CAR_t = Σ AR_s  for s≤t     (cumulative abnormal return)

    Returns
    -------
    dict[asset] → DataFrame(rel_day, AR_bp, CAR_bp, t_AR)
        AR_bp  : daily abnormal change in basis points
        CAR_bp : cumulative sum of AR_bp
        t_AR   : t-stat of daily AR  (AR / σ_est)

    Notes
    -----
    - Yields → changes in bp  (×100 applied to % series).
    - Oil   → log returns ×100  (percent-like, but scale-consistent).
    - σ is estimated on the estimation window with ddof=1.
    - Zero-change days due to fwd-fill are **excluded** from σ estimation
      to avoid downward bias.
    """
    idx     = df.index
    t0_pos  = idx.searchsorted(t0)
    est_win = get_biz_window_idx(t0, w_est[0], w_est[1], idx)
    evt_win = get_biz_window_idx(t0, w_evt[0], w_evt[1], idx)

    out = {}

    for asset in assets:
        if asset not in df.columns:
            continue

        lvl = df[asset].dropna()

        # ── Returns ─────────────────────────────────────────────────────────
        if asset == 'oil':
            # Log returns ×100 for the price series
            ret = np.log(lvl / lvl.shift(1)) * 100
        else:
            # Basis-point change for yield series
            ret = lvl.diff() * 100

        # ── Estimation window ────────────────────────────────────────────────
        r_est = ret.loc[est_win].dropna()
        # Exclude zero-change days (artefact of forward-filling)
        r_est = r_est[r_est != 0]
        if r_est.empty or len(r_est) < 30:
            print(f'  ⚠  {asset}: too few estimation obs ({len(r_est)}), skipping')
            continue

        mu    = r_est.mean()
        sigma = r_est.std(ddof=1)

        # ── Event window ─────────────────────────────────────────────────────
        r_evt = ret.reindex(evt_win).fillna(0).values
        AR    = r_evt - mu
        CAR   = np.cumsum(AR)
        rel   = np.array([idx.searchsorted(d) - t0_pos for d in evt_win])
        t_AR  = AR / (sigma + 1e-10)

        out[asset] = (
            pd.DataFrame({'rel_day': rel,
                          'AR_bp':   AR,
                          'CAR_bp':  CAR,
                          't_AR':    t_AR,
                          'mu_bp':   mu,
                          'sigma_bp': sigma})
            .set_index('rel_day')
            .sort_index()
        )

    return out


print('✓ Event study functions defined')

✓ Event study functions defined


In [ ]:
#  5-B.  Run for all events
# ────────────────────────────────────────────────────────────────────────────

results = {
    lbl: run_event_study(df_full, t0, ASSETS_ALL)
    for lbl, t0 in EVENTS.items()
}

print('\n✓ Event study complete')
for lbl, res in results.items():
    print(f'  {lbl}: {len(res)} assets processed')


✓ Event study complete
  Ukraine (23/02/2022): 12 assets processed
  Iran Escalation (27/02/2026): 12 assets processed


---
## 6. CAAR Plots  *(Cumulative Abnormal Abnormal Returns, in bp)*

In [ ]:
#  Plot 3 — CAAR in basis points, all events on same chart per asset
#           + ±1.96σ confidence band (built from estimation window σ)
# ────────────────────────────────────────────────────────────────────────────

def plot_caar_asset(asset_code: str, results: dict,
                    w_evt=W_EVT) -> go.Figure:
    """
    One figure per asset:
      - Line: CAR_bp rebased to 0 at t0  (so both events start at the same baseline)
      - Shaded band: ±1.96 * σ * sqrt(L)  where L = number of days since t0
        (expanding confidence band under the constant-mean null)
    """
    fig = go.Figure()

    for evt_label, evt_res in results.items():
        if asset_code not in evt_res:
            continue

        df_a   = evt_res[asset_code].copy()
        color  = COLORS_EVENTS[evt_label]
        sigma  = df_a['sigma_bp'].iloc[0]      # constant over estimation window

        # Rebase CAR to 0 at rel_day == 0
        car_t0 = df_a.loc[0, 'CAR_bp'] if 0 in df_a.index else 0.0
        df_a['CAR_rebased'] = df_a['CAR_bp'] - car_t0

        # Expanding 95% CI band: ±1.96·σ·√L  (L = days since t0, min 1)
        L = np.maximum(df_a.index.values - df_a.index[0] + 1, 1)
        band_width = 1.96 * sigma * np.sqrt(L)

        x = df_a.index.tolist()
        y = df_a['CAR_rebased'].values

        # Shaded band
        fig.add_trace(go.Scatter(
            x=x + x[::-1],
            y=(y + band_width).tolist() + (y - band_width).tolist()[::-1],
            fill='toself',
            fillcolor=color.replace(')', ',0.12)').replace('rgb', 'rgba')
                        if color.startswith('rgb')
                        else f'rgba(100,100,100,0.10)',
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip',
        ))

        # CAR line
        fig.add_trace(go.Scatter(
            x=x, y=y,
            mode='lines+markers',
            name=evt_label,
            line=dict(color=color, width=2.5),
            marker=dict(size=4),
            hovertemplate='Day %{x}<br>CAR = %{y:.1f} bp<extra></extra>',
        ))

    # Reference lines
    fig.add_hline(y=0, line=dict(color='black', width=1, dash='dash'))
    fig.add_vline(x=0, line=dict(color='gray',  width=1.5, dash='dash'))

    unit = 'log-ret ×100' if asset_code == 'oil' else 'bp'
    fig.update_layout(
        title=dict(
            text=f'CAAR — {LABELS.get(asset_code, asset_code)}',
            font=dict(size=16)
        ),
        xaxis_title='Business days relative to event (0 = event date)',
        yaxis_title=f'Cumulative Abnormal Return ({unit})',
        height=480,
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', y=-0.18, x=0),
    )
    return fig


for asset in ASSETS_ALL:
    fig = plot_caar_asset(asset, results)
    fname = f'plot3_CAAR_{asset}.html'
    fig.write_html(fname)
    fig.show()
    print(f'✓ CAAR plot saved → {fname}')

✓ CAAR plot saved → plot3_CAAR_bund2y.html


✓ CAAR plot saved → plot3_CAAR_bund5y.html


✓ CAAR plot saved → plot3_CAAR_bund10y.html


✓ CAAR plot saved → plot3_CAAR_ea2Y.html


✓ CAAR plot saved → plot3_CAAR_ea5Y.html


...

✓ CAAR plot saved → plot3_CAAR_fwd5y5y_ea.html


✓ CAAR plot saved → plot3_CAAR_slope_10y2y.html


✓ CAAR plot saved → plot3_CAAR_oil.html


✓ CAAR plot saved → plot3_CAAR_ea1Y.html


✓ CAAR plot saved → plot3_CAAR_fwd1y1y_ea.html


In [ ]:
#  Plot 3b — Summary grid: all yield assets in one figure (one subplot each)
# ────────────────────────────────────────────────────────────────────────────

YIELD_PANEL_ASSETS = ['bund2y', 
                      'bund10y',
                      'ea10Y',
                      'slope_10y2y',
                      'fwd5y5y_ea',
                      'fwd1y1y_ea',
                      'oil']

fig_grid = make_subplots(
    rows=4, cols=2,
    subplot_titles=[LABELS.get(a, a) for a in YIELD_PANEL_ASSETS] + [''],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

positions = [(1,1),(1,2),(2,1),(2,2),(3,1), (3,2), (4,1)]

for pos, asset in zip(positions, YIELD_PANEL_ASSETS):
    row, col = pos
    for evt_label, evt_res in results.items():
        if asset not in evt_res:
            continue
        df_a   = evt_res[asset].copy()
        color  = COLORS_EVENTS[evt_label]
        car_t0 = df_a.loc[0, 'CAR_bp'] if 0 in df_a.index else 0.0
        df_a['CAR_rebased'] = df_a['CAR_bp'] - car_t0
        fig_grid.add_trace(
            go.Scatter(
                x=df_a.index, y=df_a['CAR_rebased'],
                mode='lines', name=evt_label,
                line=dict(color=color, width=2),
                showlegend=(pos == (1,1)),  # legend only once
                legendgroup=evt_label,
            ),
            row=row, col=col
        )
    fig_grid.add_hline(y=0, line=dict(color='black', width=0.8, dash='dash'),
                       row=row, col=col)
    fig_grid.add_vline(x=0, line=dict(color='gray', width=1.2, dash='dash'),
                       row=row, col=col)

fig_grid.update_layout(
    title=dict(text='CAAR Summary — Key Assets across Both Events (bp)',
               font=dict(size=17)),
    height=900, template='plotly_white', hovermode='x unified',
    legend=dict(orientation='h', y=-0.06, x=0),
)
fig_grid.update_xaxes(title_text='Relative day')
fig_grid.update_yaxes(title_text='CAR (bp)')

fig_grid.write_html('plot3b_CAAR_summary_grid.html')
fig_grid.show()
print('✓ CAAR summary grid saved → plot3b_CAAR_summary_grid.html')

✓ CAAR summary grid saved → plot3b_CAAR_summary_grid.html


---
## 7. Statistical Inference

We report four tests per asset × event:

| Test | Formula | H₀ |
|---|---|---|
| **t-AR(0)** | `AR(0) / σ` | No abnormal move on the event day |
| **t-CAR[0,5]** | `CAR(0,5) / (σ√6)` | No cumulative effect over first week |
| **t-CAR[0,20]** | `CAR(0,20) / (σ√21)` | No cumulative effect over first month |
| **Sign test CAR[0,20]** | Binomial on sign of daily ARs | ARs are symmetrically distributed |


In [ ]:
#  7-A.  Build statistics table
# ────────────────────────────────────────────────────────────────────────────

def significance_stars(p: float) -> str:
    if   p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    return ''


rows = []

for evt_label, evt_res in results.items():
    for asset_code, df_a in evt_res.items():
        sigma = df_a['sigma_bp'].iloc[0]
        if sigma < 1e-6:
            continue

        # ── t-stat on AR at day 0 ────────────────────────────────────────────
        if 0 in df_a.index:
            ar0   = df_a.loc[0, 'AR_bp']
            t_ar0 = ar0 / sigma
            p_ar0 = 2 * stats.t.sf(abs(t_ar0), df=W_EST[1] - W_EST[0])
        else:
            ar0, t_ar0, p_ar0 = np.nan, np.nan, np.nan

        # ── CAR[0, +5] ───────────────────────────────────────────────────────
        evt_post5  = df_a.loc[(df_a.index >= 0) & (df_a.index <= 5)]
        L5         = len(evt_post5)
        car_5      = evt_post5['AR_bp'].sum()   # re-sum from ARs to be safe
        t_car5     = car_5 / (sigma * np.sqrt(L5)) if L5 > 0 else np.nan
        p_car5     = 2 * stats.t.sf(abs(t_car5), df=W_EST[1] - W_EST[0]) \
                     if not np.isnan(t_car5) else np.nan

        # ── CAR[0, +20] ──────────────────────────────────────────────────────
        evt_post20 = df_a.loc[(df_a.index >= 0) & (df_a.index <= 20)]
        L20        = len(evt_post20)
        car_20     = evt_post20['AR_bp'].sum()
        t_car20    = car_20 / (sigma * np.sqrt(L20)) if L20 > 0 else np.nan
        p_car20    = 2 * stats.t.sf(abs(t_car20), df=W_EST[1] - W_EST[0]) \
                     if not np.isnan(t_car20) else np.nan

        # ── Sign test on post-event ARs [0, +20] ─────────────────────────────
        post_ars   = evt_post20['AR_bp'].dropna().values
        n_pos      = int((post_ars > 0).sum())
        n_obs_sign = int((post_ars != 0).sum())
        if n_obs_sign > 0:
            # Two-sided binomial test (H0: p=0.5)
            binom_res = stats.binomtest(n_pos, n_obs_sign, 0.5, alternative='two-sided')
            p_sign    = binom_res.pvalue
        else:
            p_sign = np.nan

        rows.append({
            'Event':          evt_label,
            'Asset':          LABELS.get(asset_code, asset_code),
            'AssetCode':      asset_code,
            'μ_est (bp)':     round(df_a['mu_bp'].iloc[0], 3),
            'σ_est (bp)':     round(sigma, 3),
            'AR(0) (bp)':     round(ar0,   2),
            't-AR(0)':        round(t_ar0, 2),
            'sig AR(0)':      significance_stars(p_ar0),
            'CAR[0,5] (bp)':  round(car_5,  2),
            't-CAR[0,5]':     round(t_car5, 2),
            'sig CAR[0,5]':   significance_stars(p_car5),
            'CAR[0,20] (bp)': round(car_20,  2),
            't-CAR[0,20]':    round(t_car20, 2),
            'sig CAR[0,20]':  significance_stars(p_car20),
            'Sign-test p':    round(p_sign, 3),
            'sig Sign':       significance_stars(p_sign),
        })

stats_df = pd.DataFrame(rows)

print('\n' + '='*90)
print('EVENT STUDY — STATISTICAL INFERENCE TABLE')
print('='*90)
print('  Significance: * p<0.10  ** p<0.05  *** p<0.01')
print('='*90)

for evt in stats_df['Event'].unique():
    sub = stats_df[stats_df['Event'] == evt].set_index('Asset')
    cols_show = ['μ_est (bp)','σ_est (bp)',
                 'AR(0) (bp)','t-AR(0)','sig AR(0)',
                 'CAR[0,5] (bp)','t-CAR[0,5]','sig CAR[0,5]',
                 'CAR[0,20] (bp)','t-CAR[0,20]','sig CAR[0,20]',
                 'Sign-test p','sig Sign']
    print(f'\n── {evt} ──')
    print(sub[cols_show].to_string())


EVENT STUDY — STATISTICAL INFERENCE TABLE
  Significance: * p<0.10  ** p<0.05  *** p<0.01

── Ukraine (23/02/2022) ──
                   μ_est (bp)  σ_est (bp)  AR(0) (bp)  t-AR(0) sig AR(0)  CAR[0,5] (bp)  t-CAR[0,5] sig CAR[0,5]  CAR[0,20] (bp)  t-CAR[0,20] sig CAR[0,20]  Sign-test p sig Sign
Asset                                                                                                                                                                            
Bund 2Y                 0.236       2.524        7.76     3.08       ***         -27.42       -4.43          ***           14.03         1.21                      1.000         
Bund 5Y                 0.349       3.010        4.65     1.54                   -33.10       -4.49          ***           16.66         1.21                      0.664         
Bund 10Y                0.265       3.172        3.73     1.18                   -25.59       -3.29          ***           21.43         1.47                      0.664 

In [ ]:
#  7-B.  Styled HTML table  (colour-coded significance)
# ────────────────────────────────────────────────────────────────────────────

def color_sig(val):
    if val == '***': return 'background-color: #1a6b1a; color: white; font-weight: bold'
    if val == '**':  return 'background-color: #4caf50; color: white'
    if val == '*':   return 'background-color: #a5d6a7'
    return ''

def color_t(val):
    try:
        v = float(val)
        if abs(v) >= 2.58: return 'font-weight: bold; color: #1a6b1a'
        if abs(v) >= 1.96: return 'color: #4caf50'
    except: pass
    return ''

display_cols = [
    'Event','Asset',
    'AR(0) (bp)','t-AR(0)','sig AR(0)',
    'CAR[0,5] (bp)','t-CAR[0,5]','sig CAR[0,5]',
    'CAR[0,20] (bp)','t-CAR[0,20]','sig CAR[0,20]',
    'Sign-test p','sig Sign'
]

styled = (
    stats_df[display_cols]
    .style
    .applymap(color_sig, subset=['sig AR(0)','sig CAR[0,5]','sig CAR[0,20]','sig Sign'])
    .applymap(color_t,   subset=['t-AR(0)','t-CAR[0,5]','t-CAR[0,20]'])
    .set_caption('Event Study — Statistical Inference  |  * p<0.10  ** p<0.05  *** p<0.01')
    .format(precision=2, na_rep='—')
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color','#2c3e50'),('color','white'),
                  ('font-size','12px'),('padding','6px')]
    }])
)

display(styled)

,Event,Asset,AR(0) (bp),t-AR(0),sig AR(0),"CAR[0,5] (bp)","t-CAR[0,5]","sig CAR[0,5]","CAR[0,20] (bp)","t-CAR[0,20]","sig CAR[0,20]",Sign-test p,sig Sign
0,Ukraine (23/02/2022),Bund 2Y,7.76,3.08,***,-27.42,-4.43,***,14.03,1.21,,1.00,
1,Ukraine (23/02/2022),Bund 5Y,4.65,1.54,,-33.10,-4.49,***,16.66,1.21,,0.66,
2,Ukraine (23/02/2022),Bund 10Y,3.73,1.18,,-25.59,-3.29,***,21.43,1.47,,0.66,
3,Ukraine (23/02/2022),AAA EA 2Y,1.34,0.80,,-22.77,-5.57,***,8.13,1.06,,0.19,
4,Ukraine (23/02/2022),AAA EA 5Y,0.97,0.37,,-34.16,-5.35,***,15.06,1.26,,0.19,
5,Ukraine (23/02/2022),AAA EA 10Y,-1.73,-0.60,,-29.72,-4.20,***,15.00,1.13,,1.00,
6,Ukraine (23/02/2022),5Y5Y Fwd (Bund),2.81,0.70,,-18.06,-1.83,*,26.16,1.42,,0.19,
7,Ukraine (23/02/2022),5Y5Y Fwd (AAA EA),-4.45,-1.25,,-25.25,-2.88,***,14.93,0.91,,1.00,
8,Ukraine (23/02/2022),Slope 10Y-2Y,-3.08,-1.54,,-6.95,-1.42,,6.87,0.75,,1.00,
9,Ukraine (23/02/2022),Brent Oil ($/bbl),-0.13,-0.06,,14.61,2.92,***,20.09,2.15,**,0.66,


In [ ]:
#  7-C.  Daily AR bar chart for the event window — one panel per event
#        Bars coloured by |t| significance level
# ────────────────────────────────────────────────────────────────────────────

FOCUS_ASSETS = ['bund10y', 'ea10Y', 'fwd5y5y_ea', 'slope_10y2y', 'oil']

for evt_label, evt_res in results.items():
    n_assets = len([a for a in FOCUS_ASSETS if a in evt_res])
    fig_bar   = make_subplots(
        rows=n_assets, cols=1,
        subplot_titles=[LABELS.get(a,a) for a in FOCUS_ASSETS if a in evt_res],
        vertical_spacing=0.06,
        shared_xaxes=True,
    )

    for i, asset in enumerate([a for a in FOCUS_ASSETS if a in evt_res], 1):
        df_a   = evt_res[asset].copy()
        sigma  = df_a['sigma_bp'].iloc[0]
        t_vals = (df_a['AR_bp'] / sigma).values

        # Colour bars by significance level
        bar_colors = [
            '#1a6b1a' if abs(t) >= 2.58 else
            '#4caf50' if abs(t) >= 1.96 else
            '#a5d6a7' if abs(t) >= 1.64 else
            '#cccccc'
            for t in t_vals
        ]

        fig_bar.add_trace(
            go.Bar(
                x=df_a.index,
                y=df_a['AR_bp'].values,
                marker_color=bar_colors,
                name=LABELS.get(asset, asset),
                hovertemplate='Day %{x}<br>AR = %{y:.2f} bp<extra></extra>',
                showlegend=False,
            ),
            row=i, col=1
        )
        fig_bar.add_hline(y=0,  line=dict(color='black', width=0.8), row=i, col=1)
        fig_bar.add_vline(x=0,  line=dict(color='red',   width=1.5, dash='dot'),
                          row=i, col=1)
        # ±1.96σ threshold lines
        fig_bar.add_hline(y= 1.96*sigma, line=dict(color='#4caf50', width=0.8, dash='dot'),
                          row=i, col=1)
        fig_bar.add_hline(y=-1.96*sigma, line=dict(color='#4caf50', width=0.8, dash='dot'),
                          row=i, col=1)

    fig_bar.update_layout(
        title=dict(
            text=(f'Daily Abnormal Returns — <b>{evt_label}</b><br>'
                  '<sup>Dark green = 1%, green = 5%, light = 10%, grey = n.s. '
                  '| dashed lines = ±1.96σ</sup>'),
            font=dict(size=15)
        ),
        height=200 * n_assets + 100,
        template='plotly_white',
        hovermode='x unified',
    )
    fig_bar.update_xaxes(title_text='Business days relative to event', row=n_assets, col=1)

    slug = evt_label.replace(' ','_').replace('/','').replace('(','').replace(')','')[:30]
    fname = f'plot5_daily_AR_{slug}.html'
    fig_bar.write_html(fname)
    fig_bar.show()
    print(f'✓ Daily AR bar chart saved → {fname}')

✓ Daily AR bar chart saved → plot5_daily_AR_Ukraine_23022022.html


✓ Daily AR bar chart saved → plot5_daily_AR_Iran_Escalation_27022026.html


---
## 8. Cross-Event Comparison — Ukraine vs Iran Escalation

In [ ]:
#  Plot 6 — Side-by-side CAR[0,20] bar chart: Ukraine vs Iran
# ────────────────────────────────────────────────────────────────────────────

car20_ukraine = []
car20_iran    = []
assets_labels = []

for asset_code in ASSETS_ALL:
    ukr_row  = stats_df[(stats_df['AssetCode'] == asset_code) &
                        (stats_df['Event'].str.contains('Ukraine'))]
    iran_row = stats_df[(stats_df['AssetCode'] == asset_code) &
                        (stats_df['Event'].str.contains('Iran'))]
    if not ukr_row.empty and not iran_row.empty:
        assets_labels.append(LABELS.get(asset_code, asset_code))
        car20_ukraine.append(float(ukr_row['CAR[0,20] (bp)'].iloc[0]))
        car20_iran.append(float(iran_row['CAR[0,20] (bp)'].iloc[0]))

fig_comp = go.Figure([
    go.Bar(name='Ukraine (Feb 2022)',        x=assets_labels, y=car20_ukraine,
           marker_color=COLORS_EVENTS['Ukraine (23/02/2022)'],
           opacity=0.85,
           hovertemplate='%{x}<br>CAR[0,20] = %{y:.1f} bp<extra>Ukraine</extra>'),
    go.Bar(name='Iran Escalation (Feb 2026)', x=assets_labels, y=car20_iran,
           marker_color=COLORS_EVENTS['Iran Escalation (27/02/2026)'],
           opacity=0.85,
           hovertemplate='%{x}<br>CAR[0,20] = %{y:.1f} bp<extra>Iran</extra>'),
])

fig_comp.add_hline(y=0, line=dict(color='black', width=1))

fig_comp.update_layout(
    barmode='group',
    title=dict(
        text='CAR[0, +20 days] Comparison — Ukraine Invasion vs Iran Escalation',
        font=dict(size=16)
    ),
    xaxis_title='Asset',
    yaxis_title='Cumulative Abnormal Return [0→+20 days] (bp)',
    height=500,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.18),
)

fig_comp.write_html('plot6_CAR20_comparison.html')
fig_comp.show()
print('✓ Cross-event CAR comparison saved → plot6_CAR20_comparison.html')

✓ Cross-event CAR comparison saved → plot6_CAR20_comparison.html


In [ ]:
#  8-B.  Export full stats table to CSV
# ────────────────────────────────────────────────────────────────────────────

stats_df.to_csv('event_study_statistics.csv', index=False)
print('✓ Full statistics table exported → event_study_statistics.csv')
print(f'  Shape: {stats_df.shape}')
print('\n── All output files ───────────────────────────────────────────────')
import glob
for f in sorted(glob.glob('*.html') + glob.glob('*.csv')):
    print(f'  {f}')

✓ Full statistics table exported → event_study_statistics.csv
  Shape: (24, 16)

── All output files ───────────────────────────────────────────────
  event_study_statistics.csv
  plot1_asset_levels_2026.html
  plot2_levels_Iran_Escalation_27022026.html
  plot2_levels_Ukraine_23022022.html
  plot2_war_comparison_bund10y.html
  plot2_war_comparison_bund2y.html
  plot2_war_comparison_bund5y.html
  plot2_war_comparison_ea10Y.html
  plot2_war_comparison_ea1Y.html
  plot2_war_comparison_ea2Y.html
  plot2_war_comparison_ea5Y.html
  plot2_war_comparison_estr.html
  plot2_war_comparison_fwd1y1y_ea.html
  plot2_war_comparison_fwd5y5y_bund.html
  plot2_war_comparison_fwd5y5y_ea.html
  plot2_war_comparison_oil.html
  plot2_war_comparison_slope_10y2y.html
  plot3_CAAR_bund10y.html
  plot3_CAAR_bund1y.html
  plot3_CAAR_bund2y.html
  plot3_CAAR_bund5y.html
  plot3_CAAR_ea10Y.html
  plot3_CAAR_ea1Y.html
  plot3_CAAR_ea2Y.html
  plot3_CAAR_ea5Y.html
  plot3_CAAR_estr.html
  plot3_CAAR_fwd1y1y_bund.htm